# AmsiNet - Step 27: AmsiNet-E word agent on CVSI-2015 (zero-shot, GPU optional, about 10 minutes)
**Why.** CVSI-2015 (ICDAR 2015 video script identification competition) is a public script-ID benchmark of cropped video words. It has no lines or blocks, so it cannot test context, but it shows whether the **word agent trained on BSTD only** transfers to an independent public benchmark. Nothing is trained or tuned on CVSI.
**Protocol.** CVSI has ten scripts; nine overlap with our ten classes (English->Latin, Hindi->Devanagari, Bengali->Bengali-Assamese, Oriya->Odia, Gujarati, Punjabi->Gurmukhi, Kannada, Tamil, Telugu). Arabic is not one of our classes and is excluded; Malayalam is absent from CVSI, so predictions are restricted to the nine shared classes (the Malayalam column is dropped and the rest renormalised). The five word agents of Step 25 (`agents/effb0w_seed*.pt`) are applied unchanged, with the same letterbox as Step 25.
**Before you run it:** CVSI-2015 needs registration (research use only): https://www.ict.griffith.edu.au/cvsi2015/Dataset.php . Upload the zip (or zips) you receive to `MyDrive/amsinet/data/cvsi/`. Section 2 prints the folder layout and the detected labels; if the labels are not detected, paste that printout back and I will adapt the parser.

In [ ]:
# 1. Setup
from google.colab import drive
import os, subprocess, sys, glob, re, json
if not os.path.ismount('/content/drive'): drive.mount('/content/drive')
ROOT = '/content/drive/MyDrive/amsinet'
assert os.path.exists(f'{ROOT}/data/bstd_index.json') and glob.glob(f'{ROOT}/agents/effb0w_seed*.pt'), 'Step 25 weights (agents/effb0w_seed*.pt) are missing on this Drive'
try: import timm
except ImportError: subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'timm'], check=True); import timm
import numpy as np, pandas as pd, torch
import torchvision.transforms as T
from PIL import Image
from scipy import stats
from sklearn.metrics import f1_score, confusion_matrix
from torch.utils.data import Dataset, DataLoader
SCRIPTS = json.load(open(f'{ROOT}/data/bstd_index.json', encoding='utf-8'))['scripts']; SIDX = {s: i for i, s in enumerate(SCRIPTS)}; C = len(SCRIPTS)
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'; print('device', DEV, '| scripts', SCRIPTS)
CV_DIR, CV_LOCAL = f'{ROOT}/data/cvsi', '/content/cvsi'; os.makedirs(CV_LOCAL, exist_ok=True)
zips = sorted(glob.glob(f'{CV_DIR}/**/*.zip', recursive=True)); print('zip files found:', zips)
assert zips or glob.glob(f'{CV_DIR}/**/*.*', recursive=True), 'Put the CVSI-2015 download in ' + CV_DIR
for z in zips:
    sub = f'{CV_LOCAL}/{os.path.splitext(os.path.basename(z))[0]}'
    if not os.path.isdir(sub): os.makedirs(sub, exist_ok=True); subprocess.run(['unzip', '-q', '-o', z, '-d', sub], check=True)
if not zips: CV_LOCAL = CV_DIR

## 2. Find the images, their script and their split (prints the layout)

In [ ]:
EXT = ('.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff')
files = [p for p in glob.glob(f'{CV_LOCAL}/**/*', recursive=True) if p.lower().endswith(EXT)]
print(len(files), 'image files'); assert files, 'no image files found under ' + CV_LOCAL
rel = [os.path.relpath(p, CV_LOCAL) for p in files]
print('example paths:'); [print('  ', r) for r in rel[:6]]
print('top-level folders:', sorted({r.split(os.sep)[0] for r in rel})[:20])
LABEL = {'english': 'Latin', 'latin': 'Latin', 'hindi': 'Devanagari', 'devanagari': 'Devanagari', 'devnagari': 'Devanagari', 'bengali': 'Bengali-Assamese', 'bangla': 'Bengali-Assamese',
         'oriya': 'Odia', 'odia': 'Odia', 'gujarati': 'Gujarati', 'gujrathi': 'Gujarati', 'gujarathi': 'Gujarati', 'punjabi': 'Gurmukhi', 'gurmukhi': 'Gurmukhi', 'kannada': 'Kannada',
         'tamil': 'Tamil', 'telugu': 'Telugu', 'telegu': 'Telugu', 'arabic': 'ARABIC'}
SPLIT = {'train': 'train', 'training': 'train', 'val': 'val', 'validation': 'val', 'valid': 'val', 'test': 'test', 'testing': 'test'}
def parse(r):
    toks = [t for t in re.split(r'[^a-z]+', r.lower()) if t]
    lab = next((LABEL[t] for t in toks if t in LABEL), None); sp = next((SPLIT[t] for t in toks if t in SPLIT), None); return lab, sp
P = pd.DataFrame({'path': files, 'rel': rel})
# the download also holds SIW-13 and a 'combined' copy (CVSI + SIW-13): keep only the official CVSI-2015 folder so no image is counted twice
P = P[P.rel.map(lambda r: 'CVSI' in r.split(os.sep))].reset_index(drop=True); print('images in the CVSI folder:', len(P)); assert len(P) > 0, 'no folder named CVSI found'
P[['script', 'split']] = [parse(r) for r in P.rel]
print('detected scripts:', P.script.value_counts(dropna=False).to_dict()); print('detected splits:', P.split.value_counts(dropna=False).to_dict())
print(P.groupby(['split', 'script'], dropna=False).size().unstack(0).fillna(0).astype(int))

**Check the table above.** Every image should have a script. A missing split is fine (everything is then evaluated as one set). If many images show `None` for the script, stop and paste this output back.

In [ ]:
D = P[P.script.notna() & (P.script != 'ARABIC')].copy(); print('kept', len(D), 'of', len(P), 'images (Arabic excluded; unlabeled dropped)')
D['y'] = D.script.map(SIDX).astype(int); SHARED = sorted(D.y.unique()); print('classes used:', [SCRIPTS[i] for i in SHARED]); assert len(SHARED) == 9, 'expected 9 shared classes, found %d' % len(SHARED)
FILL = (124, 116, 104)
def letterbox(im, w, h):
    iw, ih = im.size; s = min(w / iw, h / ih); nw, nh = max(1, int(round(iw * s))), max(1, int(round(ih * s)))
    im = im.resize((nw, nh), Image.BICUBIC, reducing_gap=3.0); cv = Image.new('RGB', (w, h), FILL); cv.paste(im, ((w - nw) // 2, (h - nh) // 2)); return cv
mcfg = timm.data.resolve_data_config({}, model=timm.create_model('efficientnet_b0.ra_in1k', pretrained=False, num_classes=C)); tfm = T.Compose([T.ToTensor(), T.Normalize(mcfg['mean'], mcfg['std'])])
class WDS(Dataset):
    def __init__(s, paths): s.p = list(paths)
    def __len__(s): return len(s.p)
    def __getitem__(s, i): return tfm(letterbox(Image.open(s.p[i]).convert('RGB'), 224, 224))
@torch.no_grad()
def predict_paths(model, paths):
    model.eval(); out = []
    for x in DataLoader(WDS(paths), batch_size=256, num_workers=4):
        with torch.autocast(DEV, dtype=torch.float16, enabled=(DEV == 'cuda')): out.append(model(x.to(DEV)).float().softmax(1).cpu())
    return torch.cat(out).numpy()
PROBS = {}
for seed in range(5):
    m = timm.create_model('efficientnet_b0.ra_in1k', pretrained=False, num_classes=C).to(DEV); m.load_state_dict(torch.load(f'{ROOT}/agents/effb0w_seed{seed}.pt', map_location=DEV))
    PROBS[seed] = predict_paths(m, D.path); print('seed', seed, 'done'); del m
np.savez_compressed(f'{ROOT}/results/cvsi_effb0w_probs.npz', **{str(k): v for k, v in PROBS.items()}); D.to_csv(f'{ROOT}/results/cvsi_index.csv', index=False)

## 3. Results  -  paste this back

In [ ]:
y = D.y.values; sel_cols = np.array(SHARED)
def metrics(Pm, mask):
    p10 = Pm[mask]; p9 = p10[:, sel_cols] / p10[:, sel_cols].sum(1, keepdims=True); pred = sel_cols[p9.argmax(1)]
    return 100 * (pred == y[mask]).mean(), 100 * f1_score(y[mask], pred, average='macro', labels=list(sel_cols), zero_division=0), pred
sets = {'all images': np.ones(len(D), bool)}
for s in ('train', 'val', 'test'):
    if (D.split == s).any(): sets[f'{s} split'] = (D.split == s).values
tci = lambda v: stats.t.ppf(0.975, len(v) - 1) * np.std(v, ddof=1) / np.sqrt(len(v))
rows = []
for name, mask in sets.items():
    r = [metrics(PROBS[s], mask)[:2] for s in range(5)]; a, f = [x[0] for x in r], [x[1] for x in r]
    rows.append(dict(set=name, images=int(mask.sum()), accuracy='%.2f +/- %.2f' % (np.mean(a), tci(a)), macro_F1='%.2f +/- %.2f' % (np.mean(f), tci(f))))
display(pd.DataFrame(rows))
key = 'test split' if 'test split' in sets else 'all images'; mask = sets[key]
print(f'\nPer-script on {key} (mean of five seeds):')
pc = np.mean([f1_score(y[mask], metrics(PROBS[s], mask)[2], average=None, labels=list(sel_cols), zero_division=0) for s in range(5)], 0) * 100
cnt = np.bincount(y[mask], minlength=C)[sel_cols]; display(pd.DataFrame({'script': [SCRIPTS[i] for i in sel_cols], 'images': cnt, 'F1 %': pc.round(1)}))
cm = sum(confusion_matrix(y[mask], metrics(PROBS[s], mask)[2], labels=list(sel_cols)) for s in range(5)); cmn = 100 * cm / cm.sum(1, keepdims=True)
print('Largest confusions (% of the true script, mean of five seeds):')
for i, j in sorted([(i, j) for i in range(9) for j in range(9) if i != j], key=lambda t: -cmn[t])[:8]: print(f'  {SCRIPTS[sel_cols[i]]} -> {SCRIPTS[sel_cols[j]]}: {cmn[i, j]:.1f}%')
print('\nSame five models on BSTD test (10 classes): 91.93 accuracy / 85.68 macro-F1 on average.')